In [1]:
sh it will tell me what type of thrash it is

SyntaxError: invalid syntax (555595597.py, line 1)

In [2]:
# @title ♻️ Auto-SmartBin Dashboard & Disposal Engine {display-mode: "form"}
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from IPython.display import HTML, display
from google.colab import output

# Secure setup and data extraction from workspace variables
DATASET_ROOT = Path("dataset")
CLASS_NAMES = ['miscellaneous_trash', 'plastic', 'textile_trash', 'vegetation']
IMAGE_SIZE = (160, 160)
CONFIDENCE_THRESHOLD = 0.75

# Global session metrics for user interactivity tracking
session_metrics = {
    "total_scanned": 0,
    "plastics_count": 0,
    "vegetation_count": 0,
    "miscellaneous_count": 0,
    "textiles_count": 0,
    "accepted_count": 0,
    "manual_review_count": 0
}

# Dynamic active model resolution (Disk first, then memory variable)
active_model = None
try:
    MODEL_PATH = Path("smartbin2_outputs/smartbin2_best.keras")
    if not MODEL_PATH.exists():
        MODEL_PATH = Path("smartbin2_outputs/smartbin2_mobilenetv2.keras")
    if MODEL_PATH.exists():
        active_model = tf.keras.models.load_model(MODEL_PATH)
except Exception:
    active_model = None

# Fallback directly to kernel model memory if disk load fails
if active_model is None:
    if 'model' in globals() and globals()['model'] is not None:
        active_model = globals()['model']

# Disposal decision guidelines
DECISION_RULES = {
    "plastic": {
        "route": "Material Recovery / Plastic Recycling Facility",
        "reason": "Verify resin code (PET/HDPE). Wash out organic residues before disposal."
    },
    "textile_trash": {
        "route": "Textile Reclamation & Shredding Hub",
        "reason": "Separate wearable items for donation. Route degraded textiles to recycling shreds."
    },
    "vegetation": {
        "route": "Organic Composting Pile / Anaerobic Digester",
        "reason": "Ideal for municipal composting. Ensure no inclusion of plastics or chemical bags."
    },
    "miscellaneous_trash": {
        "route": "Sanitary Landfill / Certified Collection Hub",
        "reason": "Examine for hazard signs or e-waste batteries. Resort to landfill only if unrecyclable."
    }
}

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

def analyze_uploaded_image_py(image_data_uri):
    """Decodes base64 image data URI, runs keras inference, updates KPIs and returns JSON."""
    global session_metrics, active_model
    try:
        # Check model live state before inference
        if active_model is None:
            if 'model' in globals() and globals()['model'] is not None:
                active_model = globals()['model']
            else:
                return json.dumps({"error": "Model is not loaded. Please train or save the baseline model first."})

        # Parse Data URI
        import base64
        from io import BytesIO
        from PIL import Image

        header, encoded = image_data_uri.split(",", 1)
        data = base64.b64decode(encoded)
        img = Image.open(BytesIO(data)).convert("RGB")
        img = img.resize(IMAGE_SIZE)

        # Prepare for MobileNetV2
        img_array = tf.keras.preprocessing.image.img_to_array(img)
        img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)
        batch_img = tf.expand_dims(img_array, 0)

        # Predict
        predictions = active_model.predict(batch_img, verbose=0)[0]
        top_idx = np.argmax(predictions)
        predicted_class = CLASS_NAMES[top_idx]
        confidence = float(predictions[top_idx])

        # Update Session Metrics
        session_metrics["total_scanned"] += 1
        if confidence >= CONFIDENCE_THRESHOLD:
            session_metrics["accepted_count"] += 1
            status = "ACCEPTED"
        else:
            session_metrics["manual_review_count"] += 1
            status = "HUMAN REVIEW REQUIRED"

        if predicted_class == "plastic":
            session_metrics["plastics_count"] += 1
        elif predicted_class == "vegetation":
            session_metrics["vegetation_count"] += 1
        elif predicted_class == "textile_trash":
            session_metrics["textiles_count"] += 1
        else:
            session_metrics["miscellaneous_count"] += 1

        rule = DECISION_RULES.get(predicted_class, {
            "route": "Manual Identification Pipeline",
            "reason": "No specific matching rules found."
        })

        # Build probabilities dictionary
        probs = {CLASS_NAMES[i]: float(predictions[i]) for i in range(len(CLASS_NAMES))}

        return json.dumps({
            "class": predicted_class.upper(),
            "confidence": confidence,
            "status": status,
            "route": rule["route"],
            "reason": rule["reason"],
            "probs": probs,
            "metrics": session_metrics
        })
    except Exception as e:
        return json.dumps({"error": str(e)})

output.register_callback('analyze_uploaded_image_py', analyze_uploaded_image_py)

# Define the single, responsive and unified dashboard frontend template string
dashboard_html = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">
    <style>
        body {
            font-family: 'Inter', sans-serif;
            background-color: #f4f6f8;
            color: #333333;
            margin: 0;
            padding: 16px;
        }
        .dashboard-container {
            max-width: 1200px;
            margin: 0 auto;
        }
        .header {
            display: flex;
            align-items: center;
            justify-content: space-between;
            margin-bottom: 24px;
            padding-bottom: 16px;
            border-bottom: 2px solid #e1e4e8;
        }
        .header h1 {
            margin: 0;
            font-size: 24px;
            color: #1b5e20;
            display: flex;
            align-items: center;
            gap: 8px;
        }
        .header p {
            margin: 4px 0 0 0;
            color: #666666;
            font-size: 14px;
        }
        /* KPI Summary Grid */
        .kpi-row {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(150px, 1fr));
            gap: 16px;
            margin-bottom: 24px;
        }
        .kpi-card {
            background: #ffffff;
            border-radius: 8px;
            padding: 16px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.05);
            display: flex;
            flex-direction: column;
            border-left: 4px solid #4caf50;
        }
        .kpi-card.scanned { border-left-color: #2196f3; }
        .kpi-card.accepted { border-left-color: #4caf50; }
        .kpi-card.review { border-left-color: #ff9800; }
        .kpi-title {
            font-size: 11px;
            text-transform: uppercase;
            color: #888888;
            font-weight: 600;
            letter-spacing: 0.5px;
        }
        .kpi-value {
            font-size: 24px;
            font-weight: 700;
            margin-top: 8px;
            color: #222222;
        }
        /* Workspace Grid */
        .workspace-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 24px;
            margin-bottom: 24px;
        }
        @media (max-width: 768px) {
            .workspace-grid {
                grid-template-columns: 1fr;
            }
        }
        .panel {
            background: #ffffff;
            border-radius: 8px;
            padding: 20px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.05);
            display: flex;
            flex-direction: column;
        }
        .panel-title {
            font-size: 16px;
            font-weight: 600;
            margin-top: 0;
            margin-bottom: 16px;
            color: #1b5e20;
            border-bottom: 1px solid #eeeeee;
            padding-bottom: 8px;
        }
        .upload-area {
            border: 2px dashed #b2dfdb;
            border-radius: 8px;
            padding: 30px 20px;
            text-align: center;
            cursor: pointer;
            background: #f1f8e9;
            transition: background 0.3s ease;
        }
        .upload-area:hover {
            background: #e8f5e9;
        }
        #file-input {
            display: none;
        }
        .preview-box {
            margin-top: 16px;
            text-align: center;
            max-height: 250px;
            display: flex;
            align-items: center;
            justify-content: center;
        }
        #image-preview {
            max-width: 100%;
            max-height: 230px;
            border-radius: 6px;
            display: none;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .status-badge {
            display: inline-block;
            padding: 6px 12px;
            border-radius: 20px;
            font-size: 12px;
            font-weight: 700;
            text-transform: uppercase;
            margin-top: 8px;
        }
        .status-badge.accepted {
            background-color: #e8f5e9;
            color: #2e7d32;
        }
        .status-badge.review {
            background-color: #fff3e0;
            color: #e65100;
        }
        .reco-container {
            margin-top: 16px;
            background-color: #fafafa;
            border: 1px solid #e0e0e0;
            border-radius: 6px;
            padding: 14px;
        }
        .reco-label {
            font-weight: 600;
            font-size: 13px;
            color: #555555;
            margin-bottom: 4px;
        }
        .reco-text {
            font-size: 14px;
            color: #222222;
            line-height: 1.5;
        }
        .canvas-wrapper {
            position: relative;
            flex-grow: 1;
            min-height: 250px;
            width: 100%;
        }
    </style>
</head>
<body>
    <div class="dashboard-container">
        <div class="header">
            <div>
                <h1>♻️ Auto-SmartBin Classifier & Decision Engine</h1>
                <p>Localized SPU Waste Management System — St Paul's University</p>
            </div>
        </div>

        <!-- KPI Row -->
        <div class="kpi-row">
            <div class="kpi-card scanned">
                <span class="kpi-title">Total Scanned</span>
                <span class="kpi-value" id="kpi-scanned">0</span>
            </div>
            <div class="kpi-card accepted">
                <span class="kpi-title">Auto Accepted</span>
                <span class="kpi-value" id="kpi-accepted">0</span>
            </div>
            <div class="kpi-card review">
                <span class="kpi-title">Human Review Required</span>
                <span class="kpi-value" id="kpi-review">0</span>
            </div>
        </div>

        <!-- Workspace Grid -->
        <div class="workspace-grid">
            <div class="panel">
                <h3 class="panel-title">Inference Input Panel</h3>
                <div class="upload-area" onclick="document.getElementById('file-input').click()">
                    <span style="font-size: 24px; display:block; margin-bottom: 8px;">📁</span>
                    <span style="font-weight: 500;">Click to upload a waste item photo</span>
                    <input type="file" id="file-input" accept="image/*" onchange="handleImageUpload(event)">
                </div>
                <div class="preview-box">
                    <img id="image-preview" src="" alt="Uploaded Waste Item Preview">
                </div>
            </div>

            <div class="panel">
                <h3 class="panel-title">Dynamic Analysis & Disposal Decision</h3>
                <div id="analysis-placeholder" style="color: #777777; font-style: italic; text-align: center; margin: auto 0;">
                    Upload an image to trigger real-time AI classification and localized action guidelines.
                </div>
                <div id="analysis-results" style="display: none;">
                    <div style="display: flex; justify-content: space-between; align-items: center;">
                        <span id="result-class" style="font-size: 20px; font-weight: 700; color: #1b5e20;">UNKNOWN</span>
                        <span id="result-status" class="status-badge accepted">ACCEPTED</span>
                    </div>

                    <div class="reco-container">
                        <div class="reco-label">🎯 Target Route</div>
                        <div class="reco-text" id="result-route">-</div>
                    </div>
                    <div class="reco-container">
                        <div class="reco-label">📋 Action Guidelines & Rationale</div>
                        <div class="reco-text" id="result-reason">-</div>
                    </div>
                </div>
            </div>
        </div>

        <!-- Chart Visualization -->
        <div class="panel" style="margin-bottom: 24px;">
            <h3 class="panel-title">Distribution and Live Class Probability Profile</h3>
            <div class="canvas-wrapper">
                <canvas id="chart-canvas"></canvas>
            </div>
        </div>
    </div>

    <script>
        var myChart = null;

        function initChart() {
            const ctx = document.getElementById('chart-canvas').getContext('2d');
            myChart = new Chart(ctx, {
                type: 'bar',
                data: {
                    labels: ['Miscellaneous', 'Plastic', 'Textile Trash', 'Vegetation'],
                    datasets: [
                        {
                            label: 'Model Prediction Confidence (%)',
                            data: [0, 0, 0, 0],
                            backgroundColor: 'rgba(76, 175, 80, 0.6)',
                            borderColor: 'rgba(76, 175, 80, 1)',
                            borderWidth: 1
                        }
                    ]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    scales: {
                        y: {
                            beginAtZero: true,
                            max: 100,
                            title: {
                                display: true,
                                text: 'Probability (%)'
                            }
                        }
                    }
                }
            });
        }

        function handleImageUpload(event) {
            const file = event.target.files[0];
            if (!file) return;

            const reader = new FileReader();
            reader.onload = function(e) {
                const base64Data = e.target.result;

                // Show image preview
                const previewImg = document.getElementById('image-preview');
                previewImg.src = base64Data;
                previewImg.style.display = 'block';

                // Hide placeholder and show dynamic layout loading values
                document.getElementById('analysis-placeholder').style.display = 'none';
                document.getElementById('analysis-results').style.display = 'block';
                document.getElementById('result-class').innerText = 'ANALYZING...';
                document.getElementById('result-route').innerText = 'Please wait...';
                document.getElementById('result-reason').innerText = 'Extracting features and scoring with MobileNetV2...';

                // Python kernel invoke
                google.colab.kernel.invokeFunction('analyze_uploaded_image_py', [base64Data], {})
                    .then(response => {
                        const data = JSON.parse(response.data['text/plain']);
                        if (data.error) {
                            alert("Error in inference: " + data.error);
                            return;
                        }

                        // Update results panel
                        document.getElementById('result-class').innerText = data.class + ' (' + (data.confidence * 100).toFixed(1) + '%)';
                        document.getElementById('result-route').innerText = data.route;
                        document.getElementById('result-reason').innerText = data.reason;

                        const badge = document.getElementById('result-status');
                        badge.innerText = data.status;
                        if (data.status === 'ACCEPTED') {
                            badge.className = 'status-badge accepted';
                        } else {
                            badge.className = 'status-badge review';
                        }

                        // Update KPIs
                        document.getElementById('kpi-scanned').innerText = data.metrics.total_scanned;
                        document.getElementById('kpi-accepted').innerText = data.metrics.accepted_count;
                        document.getElementById('kpi-review').innerText = data.metrics.manual_review_count;

                        // Update live chart values
                        const probArray = [
                            (data.probs['miscellaneous_trash'] * 100).toFixed(1),
                            (data.probs['plastic'] * 100).toFixed(1),
                            (data.probs['textile_trash'] * 100).toFixed(1),
                            (data.probs['vegetation'] * 100).toFixed(1)
                        ];
                        myChart.data.datasets[0].data = probArray;
                        myChart.update();
                    })
                    .catch(err => {
                        google.colab.kernel.invokeFunction('report_js_error', [err.message], {});
                    });
            };
            reader.readAsDataURL(file);
        }

        window.onload = function() {
            initChart();
        };

        // Fallback initialization check in case document is already loaded
        if (document.readyState === 'complete' || document.readyState === 'interactive') {
            initChart();
        }
    </script>
</body>
</html>
"""

display(HTML(dashboard_html))


ModuleNotFoundError: No module named 'google.colab'

In [6]:
# Run after the model and test_ds have been created.
model_y_prob = model.predict(test_ds, verbose=0)
model_y_prob[:5]

NameError: name 'model' is not defined

In [ ]:
# Programmatically execute setup, configuration, dataset building, training, and testing
from IPython.display import display, Javascript

# Include setup, configuration, dataset building, training, and testing in the sequential run list
cells_to_run = [
    "13d280b0", # Pip installs
    "9f6231c9", # Imports and basic setup variables
    "700c6a76", # File structure setup and image categorization
    "47acfc1a", # Dataset audit
    "f9b90916", # Data splitting, dataset creation, and initial training
    "f8ad340b", # Model head training and fine-tuning
    "79b82c87"  # Test evaluation
]

print(f"Triggering execution for {len(cells_to_run)} pipeline cells sequentially...\n")

js_code = ""
for cid in cells_to_run:
    js_code += f"""\n    var cell = document.getElementById('{cid}');\n    if (cell) {{\n        cell.querySelector('.run-button, .execute-button, button').click();\n    }}\n    """

display(Javascript(js_code))
print("Execution requests for setup, training, and testing dispatched successfully!")

Triggering execution for 7 pipeline cells sequentially...



<IPython.core.display.Javascript object>

Execution requests for setup, training, and testing dispatched successfully!


## 1. Dataset preparation
Expected structure (one folder per class):
```
dataset/
  cardboard/
  glass/
  metal/
  paper/
  plastic/
  trash/
```

In [ ]:
%pip install -q -U tensorflow scikit-learn pandas matplotlib pillow

In [ ]:
import os, json, random, hashlib, time, zipfile, platform
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
ImageFile.LOAD_TRUNCATED_IMAGES = False

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# CHANGE THIS to your dataset directory.
DATASET_ROOT = Path("dataset")
OUTPUT_DIR = Path("smartbin2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

IMAGE_SIZE = (160, 160)
BATCH_SIZE = 16
EPOCHS_HEAD = 8
EPOCHS_FINE = 5
TEST_FRACTION = 0.15
VALIDATION_FRACTION = 0.15
CONFIDENCE_THRESHOLD = 0.75  # provisional; review validation results
AUTOTUNE = tf.data.AUTOTUNE
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

In [ ]:
import shutil
from pathlib import Path

# Explicitly define variables in case preceding configuration cells haven't run
DATASET_ROOT = Path("dataset")

# Create target directories for the custom dataset
for category in ['cardboard', 'glass', 'metal', 'organic']:
    (DATASET_ROOT / category).mkdir(parents=True, exist_ok=True)

# Scan files in /content and move them to their respective folders
all_content_files = list(Path("/content").glob("*"))

for file_path in all_content_files:
    filename = file_path.name.lower()
    dest_dir = None

    # Determine destination folder based on prefix/patterns
    if "cardboard" in filename:
        dest_dir = DATASET_ROOT / "cardboard"
    elif "glass" in filename:
        dest_dir = DATASET_ROOT / "glass"
    elif "metal" in filename:
        dest_dir = DATASET_ROOT / "metal"
    elif "organic" in filename or "food organics" in filename:
        dest_dir = DATASET_ROOT / "organic"

    if dest_dir and file_path.is_file():
        shutil.copy(file_path, dest_dir / file_path.name)

print("Dataset organization completed. Subfolders inside 'dataset/':")
for d in DATASET_ROOT.iterdir():
    if d.is_dir():
        print(f"- {d.name}: {len(list(d.glob('*')))} files")

In [ ]:
import shutil
from pathlib import Path

# Define the paths for the new categories based on the uploaded files
DATASET_ROOT = Path("dataset")
categories = ["plastic", "textile_trash", "vegetation", "miscellaneous_trash"]

for cat in categories:
    (DATASET_ROOT / cat).mkdir(parents=True, exist_ok=True)

# Scan files from /content and copy them to appropriate subfolders
content_dir = Path("/content")
for file_path in content_dir.glob("*.jpg"):
    filename = file_path.name.lower()
    dest = None
    if "plastic_" in filename:
        dest = DATASET_ROOT / "plastic"
    elif "textile trash_" in filename:
        dest = DATASET_ROOT / "textile_trash"
    elif "vegetation_" in filename:
        dest = DATASET_ROOT / "vegetation"
    elif "miscellaneous trash_" in filename:
        dest = DATASET_ROOT / "miscellaneous_trash"

    if dest:
        shutil.copy(file_path, dest / file_path.name)

print("Organized newly uploaded files. Subfolder counts:")
for d in DATASET_ROOT.iterdir():
    if d.is_dir():
        print(f"- {d.name}: {len(list(d.glob('*')))} files")

Organized newly uploaded files. Subfolder counts:
- plastic: 655 files
- textile_trash: 318 files
- miscellaneous_trash: 369 files
- vegetation: 435 files


In [ ]:
# Perform audit, dynamic class discovery, and create dataframes
CLASS_NAMES = sorted(p.name for p in DATASET_ROOT.iterdir() if p.is_dir())
NUM_CLASSES = len(CLASS_NAMES)

records = []
for class_name in CLASS_NAMES:
    for path in (DATASET_ROOT / class_name).glob("*.jpg"):
        try:
            with Image.open(path) as im:
                width, height = im.size
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            records.append({
                "path": str(path.resolve()), "class_name": class_name,
                "width": width, "height": height, "sha256": digest
            })
        except Exception as e:
            pass

audit_df = pd.DataFrame(records)
print(f"Audited {len(audit_df)} images across {NUM_CLASSES} classes: {CLASS_NAMES}")

NameError: name 'pd' is not defined

In [ ]:
# Perform Stratified Split and Pipeline setup
groups_df = audit_df.groupby("sha256").agg(path=("path","first"), class_name=("class_name","first")).reset_index()

train_val_df, test_df = train_test_split(groups_df, test_size=TEST_FRACTION, random_state=SEED, stratify=groups_df["class_name"])
val_relative = VALIDATION_FRACTION / (1 - TEST_FRACTION)
train_df, val_df = train_test_split(train_val_df, test_size=val_relative, random_state=SEED, stratify=train_val_df["class_name"])

class_to_index = {name: i for i, name in enumerate(CLASS_NAMES)}
train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

print(f"Splits completed: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

NameError: name 'audit_df' is not defined

In [ ]:
# Re-instantiate model with exact class count, train and fine-tune
base_model = tf.keras.applications.MobileNetV2(input_shape=IMAGE_SIZE+(3,), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE+(3,))
x = augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Training the Head...")
history_head = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=callbacks)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Fine-tuning later layers...")
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, callbacks=callbacks)

NameError: name 'tf' is not defined

In [ ]:
# Test and evaluate accuracy
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print(f"\nFine-tuned Test Accuracy: {test_accuracy * 100:.2f}%")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

NameError: name 'model' is not defined

In [ ]:
# Perform the dataset audit to discover and register all structured images
import pandas as pd
import hashlib
from PIL import Image
from pathlib import Path

DATASET_ROOT = Path("dataset")
OUTPUT_DIR = Path("smartbin2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

try:
    CLASS_NAMES = sorted(p.name for p in DATASET_ROOT.iterdir() if p.is_dir())
    if len(CLASS_NAMES) < 2:
        raise ValueError("At least two class subfolders are required.")

    records, unreadable, hash_to_paths = [], [], {}
    for class_name in CLASS_NAMES:
        for path in (DATASET_ROOT / class_name).rglob("*"):
            if not path.is_file() or path.suffix.lower() not in SUPPORTED_EXTENSIONS:
                continue
            try:
                with Image.open(path) as im:
                    im.verify()
                with Image.open(path) as im:
                    width, height, mode = im.width, im.height, im.mode
                digest = hashlib.sha256(path.read_bytes()).hexdigest()
                records.append({
                    "path": str(path.resolve()), "class_name": class_name,
                    "width": width, "height": height, "mode": mode, "sha256": digest
                })
                hash_to_paths.setdefault(digest, []).append(str(path))
            except Exception as exc:
                unreadable.append({"path": str(path), "error": str(exc)})

    audit_df = pd.DataFrame(records)
    unreadable_df = pd.DataFrame(unreadable)
    if audit_df.empty:
        raise ValueError("No readable images found in the organized directories.")

    print("Classes discovered:", CLASS_NAMES)
    print(f"Readable: {len(audit_df)} | Unreadable: {len(unreadable_df)}")
except Exception as e:
    print("Error in dataset audit step:", e)

Classes discovered: ['miscellaneous_trash', 'plastic', 'textile_trash', 'vegetation']
Readable: 1777 | Unreadable: 0


In [ ]:
# Split data into stratified sets, build input pipeline and start training
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Fallback declarations in case they are missing from runtime
SEED = 42
TEST_FRACTION = 0.15
VALIDATION_FRACTION = 0.15
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 16
EPOCHS_HEAD = 8
AUTOTUNE = tf.data.AUTOTUNE

try:
    # Ensure audit_df is populated
    if 'audit_df' not in locals() or audit_df.empty:
        records = []
        for class_name in sorted(p.name for p in DATASET_ROOT.iterdir() if p.is_dir()):
            for path in (DATASET_ROOT / class_name).rglob("*.jpg"):
                records.append({"path": str(path.resolve()), "class_name": class_name, "sha256": path.name})
        audit_df = pd.DataFrame(records)

    groups_df = (
        audit_df.groupby("sha256")
        .agg(path=("path","first"), class_name=("class_name","first"))
        .reset_index()
    )

    train_val_df, test_df = train_test_split(
        groups_df, test_size=TEST_FRACTION, random_state=SEED,
        stratify=groups_df["class_name"]
    )
    val_relative = VALIDATION_FRACTION / (1 - TEST_FRACTION)
    train_df, val_df = train_test_split(
        train_val_df, test_size=val_relative, random_state=SEED,
        stratify=train_val_df["class_name"]
    )

    CLASS_NAMES = sorted(list(groups_df["class_name"].unique()))
    class_to_index = {name:i for i,name in enumerate(CLASS_NAMES)}
    NUM_CLASSES = len(CLASS_NAMES)

    def load_image(path, label):
        raw = tf.io.read_file(path)
        image = tf.io.decode_image(raw, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMAGE_SIZE)
        image = tf.cast(image, tf.float32)
        image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
        return image, label

    def make_dataset(frame, training=False):
        paths = frame["path"].astype(str).values
        labels = frame["class_name"].map(class_to_index).astype("int32").values
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
        return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

    train_ds = make_dataset(train_df, True)
    val_ds = make_dataset(val_df)
    test_ds = make_dataset(test_df)

    print(f"Data split complete. Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
except Exception as e:
    print("Error during split and training setup:", e)

Data split complete. Train=1243, Val=267, Test=267


In [ ]:
### Note on Dataset Audit
This section runs the file-system scan, validates image structures, computes SHA256 hashes, and filters out unreadable or unsupported files to generate a master audit DataFrame.

In [ ]:
### Note on Balance and Duplicates
This section plots the distribution of classes and screens for cross-class label conflicts to ensure reliable generalization.

In [ ]:
### Note on Stratified Split
This section splits our distinct image groups into training, validation, and test datasets in a stratified manner, avoiding any target leakage.

## 2. Discover classes and audit the dataset

In [ ]:
if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATASET_ROOT.resolve()}. "
        "Set DATASET_ROOT to the folder containing class subfolders."
    )

CLASS_NAMES = sorted(p.name for p in DATASET_ROOT.iterdir() if p.is_dir())
if len(CLASS_NAMES) < 2:
    raise ValueError("At least two class subfolders are required.")

records, unreadable, hash_to_paths = [], [], {}
for class_name in CLASS_NAMES:
    for path in (DATASET_ROOT / class_name).rglob("*"):
        if not path.is_file() or path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        try:
            with Image.open(path) as im:
                im.verify()
            with Image.open(path) as im:
                width, height, mode = im.width, im.height, im.mode
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            records.append({
                "path": str(path.resolve()), "class_name": class_name,
                "width": width, "height": height, "mode": mode, "sha256": digest
            })
            hash_to_paths.setdefault(digest, []).append(str(path))
        except Exception as exc:
            unreadable.append({"path": str(path), "error": str(exc)})

audit_df = pd.DataFrame(records)
unreadable_df = pd.DataFrame(unreadable)
if audit_df.empty:
    raise ValueError("No readable images found.")

print("Classes:", CLASS_NAMES)
print("Readable images:", len(audit_df), "| Unreadable:", len(unreadable_df))
display(audit_df.groupby("class_name").size().rename("images").to_frame())
display(unreadable_df.head(20))

audit_df.to_csv(OUTPUT_DIR / "dataset_audit.csv", index=False)
unreadable_df.to_csv(OUTPUT_DIR / "unreadable_images.csv", index=False)

In [ ]:
audit_df["class_name"].value_counts().sort_index().plot(kind="bar", figsize=(10,4))
plt.title("Image count per class")
plt.xlabel("Class"); plt.ylabel("Images"); plt.xticks(rotation=35, ha="right")
plt.tight_layout(); plt.show()

duplicate_groups = {h:p for h,p in hash_to_paths.items() if len(p)>1}
print("Exact duplicate groups:", len(duplicate_groups))
for digest, paths in list(duplicate_groups.items())[:10]:
    print(digest[:12], paths[:5])

hash_class_counts = audit_df.groupby("sha256")["class_name"].nunique()
if (hash_class_counts > 1).any():
    raise ValueError("Identical images have conflicting labels. Resolve before training.")

## 3. Create train, validation and test splits

In [ ]:
groups_df = (
    audit_df.groupby("sha256")
    .agg(path=("path","first"), class_name=("class_name","first"),
         duplicate_count=("path","count"))
    .reset_index()
)

train_val_df, test_df = train_test_split(
    groups_df, test_size=TEST_FRACTION, random_state=SEED,
    stratify=groups_df["class_name"]
)
val_relative = VALIDATION_FRACTION / (1 - TEST_FRACTION)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_relative, random_state=SEED,
    stratify=train_val_df["class_name"]
)

for name, frame in [("train",train_df),("validation",val_df),("test",test_df)]:
    print(f"\n{name}: {len(frame)} images")
    display(frame["class_name"].value_counts().sort_index().to_frame("count"))

class_to_index = {name:i for i,name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

## 4. TensorFlow input pipeline

In [ ]:
def load_image(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

def make_dataset(frame, training=False):
    paths = frame["path"].astype(str).values
    labels = frame["class_name"].map(class_to_index).astype("int32").values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

# Visualize some training samples
try:
    images, labels = next(iter(train_ds))
    plt.figure(figsize=(10,8))
    for i in range(min(9, len(images))):
        ax = plt.subplot(3,3,i+1)
        plt.imshow(np.clip((images[i].numpy()+1)/2, 0, 1))
        plt.title(CLASS_NAMES[int(labels[i].numpy())]); plt.axis("off")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("Could not plot sample batch:", e)

## 5. Build MobileNetV2 transfer-learning model

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10)
], name="augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE+(3,), include_top=False, weights="imagenet"
)
base_model.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE+(3,))
x = augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs, name="smartbin2_mobilenetv2")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 6. Train the classification head

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
import numpy as np

# Fallback configurations
IMAGE_SIZE = (160, 160)
EPOCHS_HEAD = 8
OUTPUT_DIR = Path("smartbin2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

try:
    # Dynamic class recovery
    CLASS_NAMES = sorted(p.name for p in Path("dataset").iterdir() if p.is_dir())
    NUM_CLASSES = len(CLASS_NAMES)

    # Setup Augmentation and Base Model
    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.10)
    ], name="augmentation")

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=IMAGE_SIZE+(3,),
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False

    inputs = keras.Input(shape=IMAGE_SIZE+(3,))
    x = augmentation(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="smartbin2_mobilenetv2")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(
            OUTPUT_DIR / "smartbin2_best.keras", monitor="val_loss", save_best_only=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6
        )
    ]

    print("Starting Model Head Training...")
    history_head = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=callbacks
    )
except Exception as e:
    print("Training step error:", e)

Starting Model Head Training...
Epoch 1/8
78/78 ━━━━━━━━━━━━━━━━━━━━ 49s 541ms/step - accuracy: 0.6307 - loss: 0.9714 - val_accuracy: 0.7790 - val_loss: 0.5363 - learning_rate: 0.0010
Epoch 2/8


KeyboardInterrupt: 

## 7. Fine-tune later MobileNetV2 layers

In [ ]:
base_model.trainable = True
fine_tune_at = max(0, len(base_model.layers)-30)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
history_fine = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, callbacks=callbacks
)

## 8. Learning curves

In [ ]:
hist = {}
for key in set(history_head.history) | set(history_fine.history):
    hist[key] = history_head.history.get(key, []) + history_fine.history.get(key, [])
history_df = pd.DataFrame(hist)
display(history_df.tail())

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(history_df["accuracy"],label="train")
axes[0].plot(history_df["val_accuracy"],label="validation")
axes[0].set_title("Accuracy"); axes[0].legend()
axes[1].plot(history_df["loss"],label="train")
axes[1].plot(history_df["val_loss"],label="validation")
axes[1].set_title("Loss"); axes[1].legend()
plt.tight_layout(); plt.show()

## 9. Final test evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
y_true = np.concatenate([y.numpy() for _,y in test_ds])
y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print("Test loss:", round(float(test_loss),4))
print("Test accuracy:", round(float(test_accuracy),4))
print(classification_report(
    y_true, y_pred, labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES, zero_division=0
))

report = classification_report(
    y_true, y_pred, labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES, output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report).transpose()
display(report_df)
report_df.to_csv(OUTPUT_DIR / "classification_report.csv")

In [ ]:
cm = confusion_matrix(y_true,y_pred,labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(max(8,NUM_CLASSES),max(6,NUM_CLASSES*0.7)))
ConfusionMatrixDisplay(cm,display_labels=CLASS_NAMES).plot(
    ax=ax,xticks_rotation=45,colorbar=False
)
plt.title("Smart Bin 2.0 — Test Confusion Matrix")
plt.tight_layout(); plt.show()

## 10. Error analysis

In [ ]:
try:
    test_paths = test_df["path"].tolist()
    errors = []
    for i,(actual,predicted) in enumerate(zip(y_true,y_pred)):
        if actual != predicted:
            errors.append({
                "path":test_paths[i],
                "actual":CLASS_NAMES[int(actual)],
                "predicted":CLASS_NAMES[int(predicted)],
                "confidence":float(np.max(y_prob[i]))
            })
    errors_df = pd.DataFrame(errors)
    print("Misclassified images:",len(errors_df))
    display(errors_df.head(30))
    errors_df.to_csv(OUTPUT_DIR/"misclassified_test_images.csv",index=False)
except NameError as ne:
    print("Error Analysis could not run because some required variables (like 'test_df' or 'y_true') are missing in the current runtime state.")
    print("Please make sure to execute all previous cells starting from '1. Dataset preparation' sequentially.")
except Exception as e:
    print("An error occurred during error analysis:", e)

Error Analysis could not run because some required variables (like 'test_df' or 'y_true') are missing in the current runtime state.
Please make sure to execute all previous cells starting from '1. Dataset preparation' sequentially.


## 11. Prediction with confidence and human-review flag

In [ ]:
def predict_image_class(image_path, confidence_threshold=CONFIDENCE_THRESHOLD):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(image_path)
    image = keras.utils.load_img(image_path,target_size=IMAGE_SIZE)
    array = keras.utils.img_to_array(image)
    array = tf.keras.applications.mobilenet_v2.preprocess_input(array)
    scores = model.predict(np.expand_dims(array,0),verbose=0)[0]
    idx = int(np.argmax(scores))
    confidence = float(scores[idx])
    return {
        "predicted_class":CLASS_NAMES[idx],
        "confidence":confidence,
        "status":"PREDICTED" if confidence >= confidence_threshold else "HUMAN_REVIEW_REQUIRED",
        "scores":{CLASS_NAMES[i]:float(scores[i]) for i in range(NUM_CLASSES)}
    }

## 12. Smart Bin taxonomy register

In [ ]:
TAXONOMY = [
    {"category":"cardboard","family":"paper_and_cardboard","model_class":"cardboard"},
    {"category":"glass","family":"glass","model_class":"glass"},
    {"category":"metal","family":"metal","model_class":"metal"},
    {"category":"paper","family":"paper_and_cardboard","model_class":"paper"},
    {"category":"plastic","family":"plastics","model_class":"plastic"},
    {"category":"trash","family":"residual","model_class":"trash"},
    {"category":"electronic_waste","family":"e_waste","model_class":None},
    {"category":"battery","family":"batteries","model_class":None},
    {"category":"organic_food_waste","family":"organic","model_class":None},
    {"category":"sanitary_waste","family":"sanitary","model_class":None},
    {"category":"diaper","family":"sanitary","model_class":None},
    {"category":"circuit_board","family":"e_waste_components","model_class":None},
    {"category":"cable","family":"e_waste_components","model_class":None}
]
taxonomy_df = pd.DataFrame(TAXONOMY)
display(taxonomy_df)
taxonomy_df.to_csv(OUTPUT_DIR/"smartbin_taxonomy_starter.csv",index=False)

## 13. Separate waste-management decision-support rules

In [ ]:
DECISION_RULES = {
    "cardboard": {
        "route":"recycling_assessment",
        "review":False,
        "reason":"Check cleanliness, dryness and local recycler acceptance."
    },
    "paper": {
        "route":"recycling_assessment",
        "review":False,
        "reason":"Check contamination, coating and recycler acceptance."
    },
    "glass": {
        "route":"material_recovery_assessment",
        "review":False,
        "reason":"Check glass type, breakage and local collection requirements."
    },
    "metal": {
        "route":"material_recovery_assessment",
        "review":False,
        "reason":"Confirm material type and facility acceptance."
    },
    "plastic": {
        "route":"material_recovery_assessment",
        "review":False,
        "reason":"Resin type, contamination and facility acceptance need confirmation."
    },
    "trash": {
        "route":"manual_sorting_or_residual_assessment",
        "review":True,
        "reason":"Residual is a broad class; inspect before deciding final treatment."
    },
    "electronic_waste": {
        "route":"verified_e_waste_collection_or_assessment",
        "review":True,
        "reason":"Assess equipment, components and condition; use an appropriate specialist pathway."
    },
    "battery": {
        "route":"specialist_battery_handling_assessment",
        "review":True,
        "reason":"Do not infer internal condition from an image; refer uncertain or damaged batteries for specialist assessment."
    },
    "organic_food_waste": {
        "route":"organic_recovery_suitability_assessment",
        "review":False,
        "reason":"Check contamination and suitability for the available organic treatment process."
    },
    "sanitary_waste": {
        "route":"approved_sanitary_waste_handling",
        "review":True,
        "reason":"Apply current local health and waste-handling requirements."
    }
}

def recommend_management_path(classification, condition="unknown", confidence=1.0):
    rule = DECISION_RULES.get(classification)
    if rule is None:
        return {
            "status":"HUMAN_REVIEW_REQUIRED",
            "recommended_next_step":"manual_identification_and_assessment",
            "reason":"No reviewed rule exists for this category."
        }
    high_risk = classification in {"battery","electronic_waste","sanitary_waste"}
    low_confidence = confidence < CONFIDENCE_THRESHOLD
    review = rule["review"] or high_risk or low_confidence or condition == "unknown"
    return {
        "status":"HUMAN_REVIEW_REQUIRED" if review else "PRELIMINARY_RECOMMENDATION",
        "classification":classification,
        "condition":condition,
        "recommended_next_step":rule["route"],
        "reason":rule["reason"],
        "facility_verification_required":True,
        "regulatory_verification_required":True
    }

print(json.dumps(
    recommend_management_path("battery",condition="unknown",confidence=0.94),
    indent=2
))

## 14. End-to-end assessment

In [ ]:
def smartbin_assess_image(image_path, condition="unknown"):
    prediction = predict_image_class(image_path)
    if prediction["status"] != "PREDICTED":
        return {
            "prediction":prediction,
            "recommendation":{
                "status":"HUMAN_REVIEW_REQUIRED",
                "recommended_next_step":"manual_identification_and_assessment",
                "reason":"Model confidence is below the configured threshold."
            }
        }
    recommendation = recommend_management_path(
        prediction["predicted_class"],condition, prediction["confidence"]
    )
    return {"prediction":prediction,"recommendation":recommendation}

## 15. Validation confidence-threshold analysis

In [ ]:
try:
    val_y = np.concatenate([y.numpy() for _,y in val_ds])
    val_scores = model.predict(val_ds,verbose=1)
    val_conf = np.max(val_scores,axis=1)
    val_pred = np.argmax(val_scores,axis=1)
    val_correct = val_pred == val_y

    rows = []
    for threshold in [0.50,0.60,0.70,0.75,0.80,0.90]:
        accepted = val_conf >= threshold
        rows.append({
            "threshold":threshold,
            "accepted_coverage":float(np.mean(accepted)),
            "accuracy_on_accepted":float(np.mean(val_correct[accepted])) if accepted.any() else np.nan,
            "human_review_rate":float(1-np.mean(accepted))
        })
    threshold_df = pd.DataFrame(rows)
    display(threshold_df)
    threshold_df.to_csv(OUTPUT_DIR/"validation_threshold_analysis.csv",index=False)
except Exception as e:
    print("Could not complete validation analysis (dataset may be empty or not yet prepared):", e)

## 16. Save model and reproducibility metadata

In [ ]:
MODEL_PATH = OUTPUT_DIR/"smartbin2_mobilenetv2.keras"
try:
    model.save(MODEL_PATH)

    with open(OUTPUT_DIR/"class_labels.json","w",encoding="utf-8") as f:
        json.dump(CLASS_NAMES,f,indent=2)
    with open(OUTPUT_DIR/"decision_rules.json","w",encoding="utf-8") as f:
        json.dump(DECISION_RULES,f,indent=2)

    metadata = {
        "project":"Smart Bin 2.0",
        "owner":"Joseph Gachumi",
        "institution":"St Paul's University",
        "version":"2.0-baseline",
        "architecture":"software-only",
        "model":"MobileNetV2 transfer learning",
        "image_size":list(IMAGE_SIZE),
        "classes":CLASS_NAMES,
        "seed":SEED,
        "test_accuracy":float(test_accuracy) if 'test_accuracy' in locals() else 0.0,
        "confidence_threshold_provisional":CONFIDENCE_THRESHOLD,
        "tensorflow_version":tf.__version__,
        "dataset_root":str(DATASET_ROOT),
        "image_count":int(len(audit_df)) if 'audit_df' in locals() else 0,
        "limitations":[
            "Visible image classification only.",
            "No chemical composition or contaminant measurement.",
            "No internal battery condition diagnosis.",
            "No automatic legal disposal certification.",
            "Decision rules require local expert and regulatory review.",
            "Uncertain and high-risk cases require human assessment."
        ]
    }
    with open(OUTPUT_DIR/"smartbin2_metadata.json","w",encoding="utf-8") as f:
        json.dump(metadata,f,indent=2)

    print("Saved model and supporting files in:",OUTPUT_DIR.resolve())
except Exception as e:
    print("Could not save model/metadata:", e)

## 17. Package downloadable model outputs

In [ ]:
ZIP_PATH = Path("smartbin2_model_package.zip")
try:
    with zipfile.ZipFile(ZIP_PATH,"w",zipfile.ZIP_DEFLATED) as z:
        for file in OUTPUT_DIR.rglob("*"):
            if file.is_file():
                z.write(file,arcname=str(Path("smartbin2_outputs")/file.relative_to(OUTPUT_DIR)))
    print("Created:",ZIP_PATH.resolve())
except Exception as e:
    print("Could not create package zip:", e)

In [ ]:
from google.colab import files

try:
    files.download('smartbin2_model_package.zip')
except Exception as e:
    print("Error downloading file:", e)

## 19. Smart Bin 2.0 Web Application Interface

In [8]:
# Install Gradio for building the web interface
%pip install -q gradio

In [10]:
import gradio as gr
import numpy as np
import tensorflow as tf
from pathlib import Path
import json

# Set paths and ensure variables exist
MODEL_PATH = Path("smartbin2_outputs/smartbin2_best.keras")
if not MODEL_PATH.exists():
    # Fallback to general saved model
    MODEL_PATH = Path("smartbin2_outputs/smartbin2_mobilenetv2.keras")

CLASS_LABELS_PATH = Path("smartbin2_outputs/class_labels.json")

# Load Model and Labels
try:
    loaded_model = tf.keras.models.load_model(MODEL_PATH)
    with open(CLASS_LABELS_PATH, "r", encoding="utf-8") as f:
        classes = json.load(f)
    print("Model and classes successfully loaded for Web App.")
except Exception as e:
    print(f"Warning: Could not load trained model or labels ({e}). Using default setup parameters.")
    loaded_model = None
    classes = ['miscellaneous_trash', 'plastic', 'textile_trash', 'vegetation']

# Provisional configuration parameters
IMG_SIZE = (160, 160)
CONF_THRESH = 0.75

DECISION_RULES = {
    "plastic": {
        "route": "Material Recovery Assessment / Plastic Recycling Bin",
        "reason": "Identify plastic resin code (e.g. PET, HDPE). Ensure container is washed and free of residual oils/chemicals."
    },
    "textile_trash": {
        "route": "Textile Reclamation & Repurposing Pathway",
        "reason": "Donate wearable garments. Non-wearable textiles should go to textile shredding facilities to avoid landfill clogging."
    },
    "vegetation": {
        "route": "Organic Material Composting & Anaerobic Digestion",
        "reason": "Suitable for municipal composting or organic household composting. Do not mix with synthetic materials."
    },
    "miscellaneous_trash": {
        "route": "Manual Sorting / Certified Landfill Transfer",
        "reason": "Check for composite materials, potential hazardous items, or batteries. Review local municipal acceptance policies."
    }
}

def predict_and_recommend(input_img):
    if input_img is None:
        return "No image uploaded.", "", "", {}

    if loaded_model is None:
        return ("Model Not Found",
                "Please ensure the training pipeline has fully completed and 'smartbin2_outputs' exists.",
                "", {})

    # Preprocess image
    img = tf.image.resize(input_img, IMG_SIZE)
    img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    img_array = tf.expand_dims(img_array, 0)

    # Run inference
    scores = loaded_model.predict(img_array)[0]
    max_idx = np.argmax(scores)
    predicted_class = classes[max_idx]
    confidence = float(scores[max_idx])

    # Build score dict for Gradio plot
    confidences = {classes[i]: float(scores[i]) for i in range(len(classes))}

    # Decision rules processing
    rule = DECISION_RULES.get(predicted_class, {
        "route": "Manual Identification Pathway",
        "reason": "Category rules not registered. Inspect manually."
    })

    if confidence >= CONF_THRESH:
        status = "🟢 PREDICTED (PASS)"
        route = rule["route"]
        reason = rule["reason"]
    else:
        status = "⚠️ HUMAN REVIEW REQUIRED (Low Confidence)"
        route = "Manual Sorting & Specialized Facility Verification"
        reason = f"The AI model's confidence ({confidence*100:.1f}%) is below the {CONF_THRESH*100:.0f}% safety threshold. Inspect visually to prevent contamination."

    return (
        f"{predicted_class.upper()} ({confidence*100:.1f}% confidence)",
        status,
        f"🧭 Recommended Route:\n{route}\n\n📋 Action Guidelines:\n{reason}",
        confidences
    )

# Design custom themes and layout
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # ♻️ Smart Bin 2.0 Web Interface
        ### MobileNetV2 Image Classifier & Rule-Based Waste Management Decision Engine
        Developed by **Joseph Gachumi** | St Paul's University
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(label="Upload Waste Image", type="numpy")
            submit_btn = gr.Button("🚀 Classify Waste Image", variant="primary")

        with gr.Column(scale=1):
            lbl_output = gr.Textbox(label="Top Predicted Class", interactive=False)
            status_output = gr.Textbox(label="System Status", interactive=False)
            reco_output = gr.Textbox(label="Decision-Support Guidelines", interactive=False, lines=6)
            scores_output = gr.Label(label="Class Probabilities")

    # Interactivity
    submit_btn.click(
        fn=predict_and_recommend,
        inputs=img_input,
        outputs=[lbl_output, status_output, reco_output, scores_output]
    )

# Launch locally without publishing a public share link.
demo.launch(share=False, debug=False, server_name="127.0.0.1", theme=gr.themes.Soft(primary_hue="green", secondary_hue="emerald"))

* Running on local URL:  http://127.0.0.1:7860


* To create a public link, set `share=True` in `launch()`.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import gradio as gr
import json
from pathlib import Path

# --- Fallback Configs ---
EPOCHS_HEAD = locals().get('EPOCHS_HEAD', 8)
EPOCHS_FINE = locals().get('EPOCHS_FINE', 5)
IMAGE_SIZE = locals().get('IMAGE_SIZE', (160, 160))
CLASS_NAMES = locals().get('CLASS_NAMES', ['miscellaneous_trash', 'plastic', 'textile_trash', 'vegetation'])
CONFIDENCE_THRESHOLD = locals().get('CONFIDENCE_THRESHOLD', 0.75)
OUTPUT_DIR = Path("smartbin2_outputs")

# --- 1. Train the Classifier Head ---
print("Step 1: Training classifier head...")
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)

# --- 2. Fine-tune Deeper Layers ---
print("Step 2: Fine-tuning deeper layers...")
base_model.trainable = True
fine_tune_at = max(0, len(base_model.layers) - 30)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=callbacks
)

# --- 3. Save Final Trained Weights ---
OUTPUT_DIR.mkdir(exist_ok=True)
FINAL_MODEL_PATH = OUTPUT_DIR / "smartbin2_best.keras"
model.save(FINAL_MODEL_PATH)
print(f"Model training completed and saved to {FINAL_MODEL_PATH}")

# --- 4. Launch Gradio Interface for Automated Classification & Disposal ---
DECISION_RULES = {
    "plastic": {
        "route": "Material Recovery / Plastic Recycling Facility",
        "reason": "Verify resin code (PET/HDPE). Wash out organic residue to prevent batch contamination."
    },
    "textile_trash": {
        "route": "Textile Reclamation & Shredding Hub",
        "reason": "Separate wearable items for donation. Route heavily degraded textiles to shredders."
    },
    "vegetation": {
        "route": "Organic Composting Pile / Anaerobic Digestion",
        "reason": "Ideal for municipal composting. Ensure zero inclusion of plastic ties or treated wood."
    },
    "miscellaneous_trash": {
        "route": "Sanitary Landfill / Waste-to-Energy sorting",
        "reason": "Examine for hazardous items or electronics. Landfill should be the last resort."
    }
}

def automatic_waste_classifier(image):
    if image is None:
        return "Please upload an image first.", "N/A", "N/A"

    # Image preparation & inference
    resized_img = tf.image.resize(image, IMAGE_SIZE)
    preprocessed_img = tf.keras.applications.mobilenet_v2.preprocess_input(resized_img)
    batch_img = tf.expand_dims(preprocessed_img, 0)

    predictions = model.predict(batch_img, verbose=0)[0]
    top_idx = np.argmax(predictions)
    pred_label = CLASS_NAMES[top_idx]
    confidence = float(predictions[top_idx])

    rule = DECISION_RULES.get(pred_label, {
        "route": "Manual Sorting Channel",
        "reason": "Unrecognized class. Inspect manually to prevent safety hazards."
    })

    guidelines = f"Recommended Route: {rule['route']}\nAction Plan: {rule['reason']}"
    status = "ACCEPTED" if confidence >= CONFIDENCE_THRESHOLD else "HUMAN REVIEW REQUIRED (Low Confidence)"

    return f"{pred_label.upper()} ({confidence*100:.1f}% Confidence)", status, guidelines

demo = gr.Interface(
    fn=automatic_waste_classifier,
    inputs=gr.Image(label="Upload Waste Image"),
    outputs=[
        gr.Textbox(label="Identified Category"),
        gr.Textbox(label="System Status"),
        gr.Textbox(label="Disposal Guidelines", lines=4)
    ],
    title="♻️ Auto-SmartBin Classifier & Disposal Engine",
    description="Drop an image here to instantly identify the waste category and get specific disposal actions without delay."
)

demo.launch(share=True)

Step 1: Training classifier head...
Epoch 1/8
78/78 ━━━━━━━━━━━━━━━━━━━━ 78s 777ms/step - accuracy: 0.9131 - loss: 0.2229 - val_accuracy: 0.8539 - val_loss: 0.4121 - learning_rate: 1.0000e-05
Epoch 2/8
78/78 ━━━━━━━━━━━━━━━━━━━━ 47s 609ms/step - accuracy: 0.9324 - loss: 0.1806 - val_accuracy: 0.8764 - val_loss: 0.3710 - learning_rate: 1.0000e-05
Epoch 3/8
78/78 ━━━━━━━━━━━━━━━━━━━━ 47s 603ms/step - accuracy: 0.9340 - loss: 0.1784 - val_accuracy: 0.8951 - val_loss: 0.3499 - learning_rate: 1.0000e-05
Epoch 4/8
78/78 ━━━━━━━━━━━━━━━━━━━━ 81s 591ms/step - accuracy: 0.9453 - loss: 0.1601 - val_accuracy: 0.8727 - val_loss: 0.3878 - learning_rate: 1.0000e-05
Epoch 5/8
78/78 ━━━━━━━━━━━━━━━━━━━━ 46s 589ms/step - accuracy: 0.9533 - loss: 0.1347 - val_accuracy: 0.8914 - val_loss: 0.3539 - learning_rate: 1.0000e-05
Epoch 6/8
78/78 ━━━━━━━━━━━━━━━━━━━━ 45s 579ms/step - accuracy: 0.9566 - loss: 0.1227 - val_accuracy: 0.8876 - val_loss: 0.3606 - learning_rate: 3.0000e-06
Step 2: Fine-tuning deeper l